In [ ]:
# ============================================================
# JAGUAR RE-IDENTIFICATION CHALLENGE - EDA (PARTE 1)
# 1) SETUP E IMPORTS
# 2) CARGA DE DATOS
# 3) VERIFICACION DE INTEGRIDAD
# ============================================================

# =========================
# 1. SETUP E IMPORTS
# =========================

import os
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from PIL import Image
from tqdm.notebook import tqdm

import matplotlib.pyplot as plt
import seaborn as sns

# Configuracion visual para pandas y graficos
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 200)

sns.set_style("whitegrid")
warnings.filterwarnings("ignore")

print("Librerias cargadas correctamente.")


In [ ]:
# =========================
# 2. CARGA DE DATOS
# =========================

# -------------------------------------------------------------------
# IMPORTANTE:
# Ajusta estos paths si Kaggle montó el dataset con una estructura distinta.
# Por ahora usamos exactamente los paths que mencionaste.
# -------------------------------------------------------------------

CSV_DIR = Path("/kaggle/input/datasets/damontoyat/train-csv-jaguars")
IMG_DIR = Path("/kaggle/input/datasets/damontoyat/train-jaguars")

print("CSV_DIR existe:", CSV_DIR.exists())
print("IMG_DIR existe:", IMG_DIR.exists())

# -------------------------------------------------------------------
# Buscar automaticamente archivos CSV dentro del folder de metadata.
# Esto es mejor que hardcodear un filename porque a veces Kaggle
# cambia la estructura interna de carpetas.
# -------------------------------------------------------------------

csv_files = list(CSV_DIR.rglob("*.csv"))

print("\nCSV files encontrados:")
for f in csv_files:
    print(" -", f)

assert len(csv_files) > 0, "No se encontro ningun archivo CSV en CSV_DIR."

# Si hay mas de uno, usamos el primero.
# Luego podemos cambiar esto si hiciera falta.
train_csv_path = csv_files[0]
print(f"\nUsando train CSV: {train_csv_path}")

# Cargar CSV
train_df = pd.read_csv(train_csv_path)

print("\nShape de train_df:", train_df.shape)
print("\nPrimeras filas:")
display(train_df.head())

print("\nColumnas:")
print(train_df.columns.tolist())



In [ ]:
# =========================
# 3. VERIFICACION DE INTEGRIDAD
# =========================

# --------------------------------------------------------
# 3.1 Verificaciones basicas del dataframe
# --------------------------------------------------------

print("=" * 70)
print("3.1 VERIFICACIONES BASICAS DEL DATAFRAME")
print("=" * 70)

# Shape esperado segun descripcion:
# ~1896 filas, 2 columnas: filename, ground_truth
print("Shape:", train_df.shape)

# Nombres de columnas esperados
expected_cols = {"filename", "ground_truth"}
found_cols = set(train_df.columns)

print("Columnas encontradas:", found_cols)
print("Columnas esperadas:", expected_cols)

missing_expected_cols = expected_cols - found_cols
extra_cols = found_cols - expected_cols

print("Columnas faltantes:", missing_expected_cols)
print("Columnas extra:", extra_cols)

assert "filename" in train_df.columns, "Falta la columna 'filename'."
assert "ground_truth" in train_df.columns, "Falta la columna 'ground_truth'."

# Tipos de datos
print("\nTipos de datos:")
print(train_df.dtypes)

# Nulos
print("\nValores nulos por columna:")
print(train_df.isnull().sum())

# Duplicados exactos de filas
exact_duplicates = train_df.duplicated().sum()
print("\nNumero de filas duplicadas exactas:", exact_duplicates)

# Duplicados de filename
filename_duplicates = train_df["filename"].duplicated().sum()
print("Numero de filenames duplicados:", filename_duplicates)

# Numero de identidades unicas
n_identities = train_df["ground_truth"].nunique()
print("Numero de identidades unicas:", n_identities)

# Numero de filenames unicos
n_unique_filenames = train_df["filename"].nunique()
print("Numero de filenames unicos:", n_unique_filenames)

In [ ]:
# --------------------------------------------------------
# 3.2 Verificacion de archivos reales en la carpeta de imagenes
# --------------------------------------------------------

print("\n" + "=" * 70)
print("3.2 VERIFICACION DE ARCHIVOS DE IMAGEN")
print("=" * 70)

# Buscar todos los PNG reales
image_files = list(IMG_DIR.rglob("*.png"))

print("Numero total de PNG encontrados en carpeta:", len(image_files))

# Extraer solo nombres de archivo (sin path completo)
image_names = set(p.name for p in image_files)
csv_names = set(train_df["filename"].astype(str).tolist())

print("Numero de filenames unicos en CSV:", len(csv_names))
print("Numero de filenames unicos en carpeta:", len(image_names))

# Archivos presentes en CSV pero no en carpeta
missing_in_folder = sorted(list(csv_names - image_names))

# Archivos presentes en carpeta pero no en CSV
extra_in_folder = sorted(list(image_names - csv_names))

print("\nImagenes listadas en CSV pero NO encontradas en carpeta:", len(missing_in_folder))
print("Imagenes presentes en carpeta pero NO listadas en CSV:", len(extra_in_folder))

if len(missing_in_folder) > 0:
    print("\nEjemplos de faltantes en carpeta:")
    print(missing_in_folder[:10])

if len(extra_in_folder) > 0:
    print("\nEjemplos de extras en carpeta:")
    print(extra_in_folder[:10])

# Assert importante: idealmente todas las imagenes del CSV deben existir
assert len(missing_in_folder) == 0, (
    "Hay imagenes referenciadas en el CSV que no existen en la carpeta."
)


In [ ]:
# --------------------------------------------------------
# 3.3 Crear path completo por imagen
# --------------------------------------------------------
# Esto nos servira luego en visualizacion y analisis tecnico.
# Como ya validamos que los filenames existen en la carpeta,
# podemos mapear filename -> path real.
# --------------------------------------------------------

print("\n" + "=" * 70)
print("3.3 MAPEO filename -> full_path")
print("=" * 70)

# Creamos un diccionario para acceso rapido
name_to_path = {p.name: str(p) for p in image_files}

train_df["full_path"] = train_df["filename"].map(name_to_path)

# Verificar que ningun full_path quede nulo
null_full_paths = train_df["full_path"].isnull().sum()
print("Numero de full_path nulos:", null_full_paths)

assert null_full_paths == 0, "Algunas imagenes no pudieron mapearse a un path real."

print("\nEjemplo de train_df con full_path:")
display(train_df.head())



In [ ]:
# --------------------------------------------------------
# 3.4 Chequeo rapido de apertura de imagenes
# --------------------------------------------------------
# No hacemos un analisis completo todavia, solo confirmamos que:
# - los archivos abren bien
# - no estan corruptos
# - PIL puede leerlos
# --------------------------------------------------------

print("\n" + "=" * 70)
print("3.4 CHEQUEO RAPIDO DE APERTURA DE IMAGENES")
print("=" * 70)

corrupted_files = []
sample_image_info = []

for i, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Verificando imagenes"):
    img_path = row["full_path"]
    try:
        with Image.open(img_path) as img:
            width, height = img.size
            mode = img.mode

            # Guardamos unas pocas muestras para inspeccion
            if len(sample_image_info) < 5:
                sample_image_info.append({
                    "filename": row["filename"],
                    "ground_truth": row["ground_truth"],
                    "width": width,
                    "height": height,
                    "mode": mode
                })

    except Exception as e:
        corrupted_files.append((row["filename"], str(e)))

print("\nNumero de archivos corruptos o no legibles:", len(corrupted_files))

if len(corrupted_files) > 0:
    print("\nEjemplos de archivos con error:")
    for item in corrupted_files[:10]:
        print(item)

assert len(corrupted_files) == 0, "Hay imagenes corruptas o no legibles."

print("\nMuestra de informacion basica de algunas imagenes:")
display(pd.DataFrame(sample_image_info))



In [ ]:
# --------------------------------------------------------
# 3.5 Resumen final de integridad
# --------------------------------------------------------

print("\n" + "=" * 70)
print("RESUMEN FINAL DE INTEGRIDAD")
print("=" * 70)

print(f"Filas en train_df: {len(train_df):,}")
print(f"Filenames unicos en train_df: {train_df['filename'].nunique():,}")
print(f"Identidades unicas: {train_df['ground_truth'].nunique():,}")
print(f"PNG encontrados en carpeta: {len(image_files):,}")
print(f"Filenames faltantes en carpeta: {len(missing_in_folder):,}")
print(f"Filenames extra en carpeta: {len(extra_in_folder):,}")
print(f"Duplicados exactos en train_df: {exact_duplicates:,}")
print(f"Duplicados de filename: {filename_duplicates:,}")
print(f"Paths nulos: {null_full_paths:,}")
print(f"Archivos corruptos: {len(corrupted_files):,}")

print("\nEstado general del dataset: OK para continuar con el EDA.")

# DISTRIBUCION DE CLASES

In [ ]:
# ============================================================
# 4.1 Conteo de imagenes por identidad
# ============================================================

# Contamos cuantas imagenes tiene cada jaguar
class_counts = (
    train_df["ground_truth"]
    .value_counts()
    .sort_values(ascending=False)
)

print("Numero de identidades:", class_counts.shape[0])
print("\nTop 10 identidades con mas imagenes:")
display(class_counts.head(10).to_frame(name="n_images"))

print("\nTop 10 identidades con menos imagenes:")
display(class_counts.tail(10).sort_values().to_frame(name="n_images"))


In [ ]:
# ============================================================
# 4.2 Resumen estadistico de la distribucion
# ============================================================

print("=" * 70)
print("RESUMEN ESTADISTICO DE IMAGENES POR IDENTIDAD")
print("=" * 70)

summary_stats = class_counts.describe()
display(summary_stats.to_frame(name="value"))

print(f"Media de imagenes por identidad: {class_counts.mean():.2f}")
print(f"Mediana de imagenes por identidad: {class_counts.median():.2f}")
print(f"Minimo de imagenes en una identidad: {class_counts.min()}")
print(f"Maximo de imagenes en una identidad: {class_counts.max()}")
print(f"Rango (max - min): {class_counts.max() - class_counts.min()}")

In [ ]:
# ============================================================
# 4.3 Tabla resumen por identidad
# ============================================================

class_dist_df = class_counts.reset_index()
class_dist_df.columns = ["ground_truth", "n_images"]

# Agregamos porcentaje respecto al total de imagenes
class_dist_df["pct_of_total"] = 100 * class_dist_df["n_images"] / len(train_df)

# Rank de frecuencia
class_dist_df["rank_by_frequency"] = np.arange(1, len(class_dist_df) + 1)

print("\nTabla resumen de distribucion de clases:")
display(class_dist_df.head(15))

In [ ]:
# ============================================================
# 4.4 Grafico de barras: numero de imagenes por identidad
# ============================================================

plt.figure(figsize=(16, 6))
plt.bar(class_dist_df["ground_truth"], class_dist_df["n_images"])
plt.title("Numero de imagenes por identidad", fontsize=14)
plt.xlabel("Identidad")
plt.ylabel("Cantidad de imagenes")
plt.xticks(rotation=90)
plt.show()


In [ ]:
# ============================================================
# 4.5 Histograma de tamanos de clase
# ============================================================

plt.figure(figsize=(8, 5))
plt.hist(class_dist_df["n_images"], bins=15)
plt.title("Distribucion del numero de imagenes por identidad", fontsize=14)
plt.xlabel("Imagenes por identidad")
plt.ylabel("Frecuencia")
plt.show()

In [ ]:
# ============================================================
# 4.6 Boxplot para visualizar dispersion y posibles extremos
# ============================================================

plt.figure(figsize=(8, 2.5))
plt.boxplot(class_dist_df["n_images"], vert=False)
plt.title("Boxplot del numero de imagenes por identidad", fontsize=14)
plt.xlabel("Imagenes por identidad")
plt.show()

In [ ]:
# ============================================================
# 4.7 Medidas simples de desbalance
# ============================================================

max_count = class_dist_df["n_images"].max()
min_count = class_dist_df["n_images"].min()
imbalance_ratio = max_count / min_count

print("=" * 70)
print("MEDIDAS DE DESBALANCE")
print("=" * 70)

print(f"Clase mas frecuente: {class_dist_df.loc[class_dist_df['n_images'].idxmax(), 'ground_truth']} ({max_count} imagenes)")
print(f"Clase menos frecuente: {class_dist_df.loc[class_dist_df['n_images'].idxmin(), 'ground_truth']} ({min_count} imagenes)")
print(f"Imbalance ratio (max/min): {imbalance_ratio:.2f}x")

# Cuantas clases tienen menos de ciertos umbrales
thresholds = [20, 30, 50, 100]
for t in thresholds:
    n_below = (class_dist_df["n_images"] < t).sum()
    print(f"Numero de identidades con menos de {t} imagenes: {n_below}")

In [ ]:
# ============================================================
# 4.8 Curva acumulada: cuantas imagenes explican las clases mas frecuentes
# ============================================================

class_dist_df["cum_n_images"] = class_dist_df["n_images"].cumsum()
class_dist_df["cum_pct_images"] = 100 * class_dist_df["cum_n_images"] / len(train_df)

plt.figure(figsize=(10, 5))
plt.plot(class_dist_df["rank_by_frequency"], class_dist_df["cum_pct_images"], marker="o")
plt.title("Cobertura acumulada de imagenes segun ranking de frecuencia", fontsize=14)
plt.xlabel("Numero de identidades mas frecuentes incluidas")
plt.ylabel("% acumulado de imagenes del train")
plt.ylim(0, 105)
plt.grid(True)
plt.show()


In [ ]:
# ============================================================
# 4.9 Resumen interpretable del desbalance
# ============================================================

top_3_pct = class_dist_df.head(3)["pct_of_total"].sum()
top_5_pct = class_dist_df.head(5)["pct_of_total"].sum()
bottom_5_pct = class_dist_df.tail(5)["pct_of_total"].sum()

print("=" * 70)
print("RESUMEN INTERPRETABLE")
print("=" * 70)

print(f"Las 3 identidades mas frecuentes representan el {top_3_pct:.2f}% del train.")
print(f"Las 5 identidades mas frecuentes representan el {top_5_pct:.2f}% del train.")
print(f"Las 5 identidades menos frecuentes representan el {bottom_5_pct:.2f}% del train.")


In [ ]:
# ============================================================
# 4.10 Tabla final ordenada para inspeccion
# ============================================================

print("\nDistribucion completa de clases:")
display(class_dist_df)

# PROPIEDADES TECNICAS DE LAS IMAGENES

In [ ]:
# ============================================================
# JAGUAR RE-IDENTIFICATION CHALLENGE - EDA (PARTE 5)
# PROPIEDADES TECNICAS DE LAS IMAGENES
# ============================================================

from PIL import Image
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm


In [ ]:
# ============================================================
# 5.1 Extraer metadata tecnica basica por imagen
# ============================================================
# Vamos a recorrer las imagenes una sola vez para extraer:
# - width
# - height
# - mode (RGB, RGBA, etc.)
# - si tiene canal alpha
# - aspect ratio
# - numero total de pixeles
# - proporcion de pixeles visibles si existe alpha
#
# Esto es mucho mas liviano que cargar todas las imagenes
# completas en memoria al mismo tiempo.
# ============================================================

image_stats = []

for _, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Extrayendo metadata tecnica"):
    img_path = row["full_path"]
    fname = row["filename"]
    identity = row["ground_truth"]

    with Image.open(img_path) as img:
        width, height = img.size
        mode = img.mode
        has_alpha = "A" in mode
        pixels = width * height
        aspect_ratio = width / height

        alpha_nonzero_ratio = np.nan
        alpha_bbox_area_ratio = np.nan
        alpha_bbox_width = np.nan
        alpha_bbox_height = np.nan

        # Si existe canal alpha, analizamos cuanto ocupa realmente
        # el objeto visible dentro de la imagen
        if has_alpha:
            alpha = np.array(img.getchannel("A"))
            visible_mask = alpha > 0

            alpha_nonzero_ratio = visible_mask.mean()

            if visible_mask.any():
                ys, xs = np.where(visible_mask)
                x_min, x_max = xs.min(), xs.max()
                y_min, y_max = ys.min(), ys.max()

                bbox_w = x_max - x_min + 1
                bbox_h = y_max - y_min + 1

                alpha_bbox_width = bbox_w
                alpha_bbox_height = bbox_h
                alpha_bbox_area_ratio = (bbox_w * bbox_h) / pixels

        image_stats.append({
            "filename": fname,
            "ground_truth": identity,
            "width": width,
            "height": height,
            "mode": mode,
            "has_alpha": has_alpha,
            "pixels": pixels,
            "aspect_ratio": aspect_ratio,
            "alpha_nonzero_ratio": alpha_nonzero_ratio,
            "alpha_bbox_width": alpha_bbox_width,
            "alpha_bbox_height": alpha_bbox_height,
            "alpha_bbox_area_ratio": alpha_bbox_area_ratio
        })

img_stats_df = pd.DataFrame(image_stats)

print("Shape de img_stats_df:", img_stats_df.shape)
display(img_stats_df.head())

In [ ]:
# ============================================================
# 5.2 Unir metadata tecnica con train_df
# ============================================================

eda_img_df = train_df.merge(
    img_stats_df,
    on=["filename", "ground_truth"],
    how="left"
)

print("Shape de eda_img_df:", eda_img_df.shape)
display(eda_img_df.head())

In [ ]:
# ============================================================
# 5.3 Resumen numerico general
# ============================================================

technical_cols = ["width", "height", "pixels", "aspect_ratio"]
print("=" * 70)
print("RESUMEN NUMERICO GENERAL")
print("=" * 70)
display(eda_img_df[technical_cols].describe().T)

print("\nDistribucion de modos de imagen:")
display(eda_img_df["mode"].value_counts().to_frame(name="count"))

print("\nDistribucion de has_alpha:")
display(eda_img_df["has_alpha"].value_counts(dropna=False).to_frame(name="count"))

In [ ]:
# ============================================================
# 5.4 Cuantos tamaños unicos hay
# ============================================================
# Esto nos dice si las imagenes ya vienen en una resolucion fija
# o si el dataset tiene variabilidad importante de tamaños.
# ============================================================

size_counts = (
    eda_img_df.groupby(["width", "height"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

print("Numero de combinaciones unicas de (width, height):", len(size_counts))
print("\nTop 15 tamaños mas frecuentes:")
display(size_counts.head(15))

In [ ]:
# ============================================================
# 5.5 Graficos de distribucion de width, height, pixels, aspect ratio
# ============================================================

plt.figure(figsize=(8, 5))
plt.hist(eda_img_df["width"], bins=25)
plt.title("Distribucion de width", fontsize=14)
plt.xlabel("Width")
plt.ylabel("Frecuencia")
plt.show()

plt.figure(figsize=(8, 5))
plt.hist(eda_img_df["height"], bins=25)
plt.title("Distribucion de height", fontsize=14)
plt.xlabel("Height")
plt.ylabel("Frecuencia")
plt.show()

plt.figure(figsize=(8, 5))
plt.hist(eda_img_df["pixels"], bins=25)
plt.title("Distribucion del numero total de pixeles", fontsize=14)
plt.xlabel("Pixels")
plt.ylabel("Frecuencia")
plt.show()

plt.figure(figsize=(8, 5))
plt.hist(eda_img_df["aspect_ratio"], bins=25)
plt.title("Distribucion de aspect ratio (width / height)", fontsize=14)
plt.xlabel("Aspect ratio")
plt.ylabel("Frecuencia")
plt.show()

In [ ]:
# ============================================================
# 5.6 Scatter width vs height
# ============================================================
# Muy util para ver si las resoluciones siguen patrones concretos
# o si hay mucha heterogeneidad.
# ============================================================

plt.figure(figsize=(7, 6))
plt.scatter(eda_img_df["width"], eda_img_df["height"], alpha=0.35)
plt.title("Width vs Height", fontsize=14)
plt.xlabel("Width")
plt.ylabel("Height")
plt.show()

In [ ]:
# ============================================================
# 5.7 Analisis del canal alpha
# ============================================================
# alpha_nonzero_ratio:
#   fraccion de pixeles con alpha > 0
#   indica aproximadamente cuanto contenido visible hay
#
# alpha_bbox_area_ratio:
#   area del bounding box del objeto visible / area total de la imagen
#   indica cuanto del lienzo ocupa el jaguar segmentado
# ============================================================

if eda_img_df["has_alpha"].any():
    print("=" * 70)
    print("ANALISIS DEL CANAL ALPHA")
    print("=" * 70)

    alpha_cols = ["alpha_nonzero_ratio", "alpha_bbox_area_ratio", "alpha_bbox_width", "alpha_bbox_height"]
    display(eda_img_df[alpha_cols].describe().T)

    plt.figure(figsize=(8, 5))
    plt.hist(eda_img_df["alpha_nonzero_ratio"].dropna(), bins=25)
    plt.title("Distribucion de alpha_nonzero_ratio", fontsize=14)
    plt.xlabel("Fraccion de pixeles visibles (alpha > 0)")
    plt.ylabel("Frecuencia")
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.hist(eda_img_df["alpha_bbox_area_ratio"].dropna(), bins=25)
    plt.title("Distribucion de alpha_bbox_area_ratio", fontsize=14)
    plt.xlabel("Area bbox visible / area total")
    plt.ylabel("Frecuencia")
    plt.show()

    plt.figure(figsize=(7, 6))
    plt.scatter(
        eda_img_df["alpha_bbox_width"],
        eda_img_df["alpha_bbox_height"],
        alpha=0.35
    )
    plt.title("Alpha bbox width vs height", fontsize=14)
    plt.xlabel("BBox width")
    plt.ylabel("BBox height")
    plt.show()
else:
    print("No se detecto canal alpha en las imagenes.")

In [ ]:
# ============================================================
# 5.8 Detectar imagenes extremas
# ============================================================
# Queremos identificar casos raros:
# - imagenes muy pequenas o muy grandes
# - aspect ratio extremo
# - muy poco contenido visible segun alpha
# ============================================================

print("=" * 70)
print("IMAGENES EXTREMAS O POTENCIALMENTE ATIPICAS")
print("=" * 70)

print("\nTop 10 imagenes con menor width:")
display(
    eda_img_df.sort_values("width", ascending=True)[
        ["filename", "ground_truth", "width", "height", "aspect_ratio", "mode"]
    ].head(10)
)

print("\nTop 10 imagenes con mayor width:")
display(
    eda_img_df.sort_values("width", ascending=False)[
        ["filename", "ground_truth", "width", "height", "aspect_ratio", "mode"]
    ].head(10)
)

print("\nTop 10 imagenes con menor height:")
display(
    eda_img_df.sort_values("height", ascending=True)[
        ["filename", "ground_truth", "width", "height", "aspect_ratio", "mode"]
    ].head(10)
)

print("\nTop 10 imagenes con mayor height:")
display(
    eda_img_df.sort_values("height", ascending=False)[
        ["filename", "ground_truth", "width", "height", "aspect_ratio", "mode"]
    ].head(10)
)

print("\nTop 10 imagenes con aspect ratio mas pequeno:")
display(
    eda_img_df.sort_values("aspect_ratio", ascending=True)[
        ["filename", "ground_truth", "width", "height", "aspect_ratio", "mode"]
    ].head(10)
)

print("\nTop 10 imagenes con aspect ratio mas grande:")
display(
    eda_img_df.sort_values("aspect_ratio", ascending=False)[
        ["filename", "ground_truth", "width", "height", "aspect_ratio", "mode"]
    ].head(10)
)

if eda_img_df["has_alpha"].any():
    print("\nTop 10 imagenes con menor alpha_nonzero_ratio:")
    display(
        eda_img_df.sort_values("alpha_nonzero_ratio", ascending=True)[
            ["filename", "ground_truth", "width", "height", "alpha_nonzero_ratio", "alpha_bbox_area_ratio"]
        ].head(10)
    )

In [ ]:
# ============================================================
# 5.9 Resumen ejecutivo tecnico
# ============================================================

print("=" * 70)
print("RESUMEN EJECUTIVO TECNICO")
print("=" * 70)

print(f"Numero total de imagenes analizadas: {len(eda_img_df):,}")
print(f"Width minimo / maximo: {eda_img_df['width'].min()} / {eda_img_df['width'].max()}")
print(f"Height minimo / maximo: {eda_img_df['height'].min()} / {eda_img_df['height'].max()}")
print(f"Pixels minimos / maximos: {eda_img_df['pixels'].min():,} / {eda_img_df['pixels'].max():,}")
print(f"Aspect ratio minimo / maximo: {eda_img_df['aspect_ratio'].min():.3f} / {eda_img_df['aspect_ratio'].max():.3f}")

print("\nModos de imagen:")
print(eda_img_df["mode"].value_counts())

print("\nPresencia de alpha:")
print(eda_img_df["has_alpha"].value_counts(dropna=False))

if eda_img_df["has_alpha"].any():
    print(f"\nPromedio alpha_nonzero_ratio: {eda_img_df['alpha_nonzero_ratio'].mean():.4f}")
    print(f"Mediana alpha_nonzero_ratio: {eda_img_df['alpha_nonzero_ratio'].median():.4f}")
    print(f"Promedio alpha_bbox_area_ratio: {eda_img_df['alpha_bbox_area_ratio'].mean():.4f}")
    print(f"Mediana alpha_bbox_area_ratio: {eda_img_df['alpha_bbox_area_ratio'].median():.4f}")

# MUESTRAS POR IDENTIDAD

In [ ]:
# ============================================================
# JAGUAR RE-IDENTIFICATION CHALLENGE - EDA (PARTE 6)
# MUESTRAS POR IDENTIDAD
# ============================================================

import math
import random
from PIL import Image
import matplotlib.pyplot as plt

In [ ]:
# ============================================================
# 6.1 Preparar tabla de frecuencias si no existe
# ============================================================

class_counts = (
    train_df["ground_truth"]
    .value_counts()
    .sort_values(ascending=False)
)

class_dist_df = class_counts.reset_index()
class_dist_df.columns = ["ground_truth", "n_images"]
class_dist_df["rank_by_frequency"] = range(1, len(class_dist_df) + 1)

display(class_dist_df.head())

In [ ]:
# ============================================================
# 6.2 Funciones auxiliares para visualizacion
# ============================================================

def load_rgba_image(img_path):
    """
    Carga una imagen usando PIL.
    La dejamos en RGBA porque asi fue almacenada en el dataset.
    """
    img = Image.open(img_path).convert("RGBA")
    return img


def show_identity_samples(identity, n=6, seed=42, figsize_scale=4):
    """
    Muestra n imagenes aleatorias de una identidad concreta.

    Parametros:
    - identity: nombre del jaguar
    - n: numero de imagenes a mostrar
    - seed: para reproducibilidad
    - figsize_scale: controla el tamano del panel
    """
    subset = train_df[train_df["ground_truth"] == identity].copy()

    n_available = len(subset)
    n_show = min(n, n_available)

    subset = subset.sample(n=n_show, random_state=seed)

    fig, axes = plt.subplots(1, n_show, figsize=(figsize_scale * n_show, figsize_scale))

    if n_show == 1:
        axes = [axes]

    for ax, (_, row) in zip(axes, subset.iterrows()):
        img = load_rgba_image(row["full_path"])
        ax.imshow(img)
        ax.set_title(row["filename"], fontsize=9)
        ax.axis("off")

    plt.suptitle(f"Identidad: {identity} | Total imagenes: {n_available}", fontsize=14)
    plt.tight_layout()
    plt.show()


def show_identity_grid(identity, n=12, seed=42, ncols=4, figsize_scale=4):
    """
    Muestra una grilla de imagenes de una identidad.
    Muy util para inspeccionar variabilidad intra-clase.
    """
    subset = train_df[train_df["ground_truth"] == identity].copy()

    n_available = len(subset)
    n_show = min(n, n_available)

    subset = subset.sample(n=n_show, random_state=seed)

    nrows = math.ceil(n_show / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_scale * ncols, figsize_scale * nrows))
    axes = axes.flatten()

    for ax in axes:
        ax.axis("off")

    for ax, (_, row) in zip(axes, subset.iterrows()):
        img = load_rgba_image(row["full_path"])
        ax.imshow(img)
        ax.set_title(row["filename"], fontsize=9)
        ax.axis("off")

    plt.suptitle(f"Identidad: {identity} | Total imagenes: {n_available}", fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# 6.3 Seleccionar identidades frecuentes, medias y raras
# ============================================================
# Queremos una muestra representativa del dataset:
# - clases frecuentes
# - clases intermedias
# - clases raras
# ============================================================

top_identities = class_dist_df.head(3)["ground_truth"].tolist()

mid_start = len(class_dist_df) // 2 - 1
mid_identities = class_dist_df.iloc[mid_start:mid_start + 3]["ground_truth"].tolist()

rare_identities = class_dist_df.tail(3)["ground_truth"].tolist()

print("Identidades frecuentes:", top_identities)
print("Identidades intermedias:", mid_identities)
print("Identidades raras:", rare_identities)

In [ ]:
# ============================================================
# 6.4 Visualizar algunas identidades frecuentes
# ============================================================

for identity in top_identities:
    show_identity_samples(identity, n=6, seed=42)

In [ ]:
# ============================================================
# 6.5 Visualizar algunas identidades intermedias
# ============================================================

for identity in mid_identities:
    show_identity_samples(identity, n=6, seed=42)

In [ ]:
# ============================================================
# 6.6 Visualizar algunas identidades raras
# ============================================================

for identity in rare_identities:
    show_identity_samples(identity, n=6, seed=42)

In [ ]:
# ============================================================
# 6.7 Grillas mas grandes para inspeccion detallada
# ============================================================
# Aqui conviene usar:
# - una identidad frecuente
# - una intermedia
# - una rara
# para comparar visualmente la variabilidad disponible.
# ============================================================

identity_frequent = top_identities[0]
identity_middle = mid_identities[0]
identity_rare = rare_identities[0]

show_identity_grid(identity_frequent, n=12, seed=42, ncols=4)
show_identity_grid(identity_middle, n=12, seed=42, ncols=4)
show_identity_grid(identity_rare, n=12, seed=42, ncols=4)

In [ ]:
# ============================================================
# 6.8 Panel resumen: una imagen por identidad
# ============================================================
# Esto sirve para ver de forma rapida diversidad inter-clase.
# Elegimos una imagen aleatoria por cada jaguar.
# ============================================================

sample_one_per_identity = (
    train_df.groupby("ground_truth", group_keys=False)
    .apply(lambda x: x.sample(1, random_state=42))
    .reset_index(drop=True)
)

n_id = len(sample_one_per_identity)
ncols = 4
nrows = math.ceil(n_id / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
axes = axes.flatten()

for ax in axes:
    ax.axis("off")

for ax, (_, row) in zip(axes, sample_one_per_identity.iterrows()):
    img = load_rgba_image(row["full_path"])
    ax.imshow(img)
    ax.set_title(row["ground_truth"], fontsize=10)
    ax.axis("off")

plt.suptitle("Una muestra por identidad", fontsize=16)
plt.tight_layout()
plt.show()

# VARIABILIDAD INTRA-CLASE

In [ ]:
# ============================================================
# JAGUAR RE-IDENTIFICATION CHALLENGE - EDA (PARTE 7)
# VARIABILIDAD INTRA-CLASE
# ============================================================

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# ------------------------------------------------------------
# Asumimos que ya existen:
# - train_df
# - class_dist_df
# - full_path en train_df
# ------------------------------------------------------------

print("Numero total de identidades:", train_df["ground_truth"].nunique())
print("Numero total de imagenes:", len(train_df))

In [ ]:
# ============================================================
# 7.1 Seleccionar identidades para analisis
# ============================================================
# Elegimos varias identidades:
# - 3 frecuentes
# - 3 intermedias
# - 3 raras
#
# Esto nos permite comparar la variabilidad visual entre
# clases con distinta cantidad de imagenes.
# ============================================================

class_counts = train_df["ground_truth"].value_counts().sort_values(ascending=False)
class_dist_df = class_counts.reset_index()
class_dist_df.columns = ["ground_truth", "n_images"]

top_identities = class_dist_df.head(3)["ground_truth"].tolist()

mid_start = len(class_dist_df) // 2 - 1
mid_identities = class_dist_df.iloc[mid_start:mid_start + 3]["ground_truth"].tolist()

rare_identities = class_dist_df.tail(3)["ground_truth"].tolist()

selected_identities = top_identities + mid_identities + rare_identities

print("Frecuentes:", top_identities)
print("Intermedias:", mid_identities)
print("Raras:", rare_identities)

In [ ]:
# ============================================================
# 7.2 Funciones auxiliares de visualizacion
# ============================================================

def load_rgba_image(img_path):
    return Image.open(img_path).convert("RGBA")


def show_identity_grid(identity, n=16, seed=42, ncols=4, figsize_scale=4):
    """
    Muestra una grilla de imagenes de una identidad.
    Util para inspeccionar variabilidad intra-clase.
    """
    subset = train_df[train_df["ground_truth"] == identity].copy()
    n_show = min(n, len(subset))
    subset = subset.sample(n=n_show, random_state=seed)

    nrows = math.ceil(n_show / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_scale * ncols, figsize_scale * nrows))
    axes = np.array(axes).reshape(-1)

    for ax in axes:
        ax.axis("off")

    for ax, (_, row) in zip(axes, subset.iterrows()):
        img = load_rgba_image(row["full_path"])
        ax.imshow(img)
        ax.set_title(row["filename"], fontsize=8)
        ax.axis("off")

    plt.suptitle(f"Identidad: {identity} | Total imagenes: {len(train_df[train_df['ground_truth'] == identity])}", fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# 7.3 Visualizacion de variabilidad intra-clase
# ============================================================
# Aqui inspeccionamos visualmente:
# - cuantas imagenes son casi iguales
# - cuanta variacion hay de pose, luz, escala y recorte
# ============================================================

for identity in selected_identities:
    show_identity_grid(identity, n=16, seed=42, ncols=4, figsize_scale=3.8)

In [ ]:
# ============================================================
# 7.4 Crear thumbnails estandarizados para medir similitud visual
# ============================================================
# Vamos a usar una medida simple y barata:
# - convertir imagenes a thumbnails pequenos
# - comparar pares dentro de la misma identidad
#
# Esto NO reemplaza embeddings reales, pero sirve en EDA
# para detectar clases muy repetitivas o muy diversas.
# ============================================================

def make_thumbnail_array(img_path, size=(64, 64), use_alpha_bbox=False):
    """
    Crea un array pequeño normalizado para comparacion visual simple.

    Parametros:
    - size: tamano final del thumbnail
    - use_alpha_bbox: si True, recorta al bounding box definido por alpha > 0

    Retorna:
    - array float32 shape (H, W, 3), en [0,1]
    """
    img = Image.open(img_path).convert("RGBA")
    arr = np.array(img)

    if use_alpha_bbox:
        alpha = arr[:, :, 3]
        mask = alpha > 0
        if mask.any():
            ys, xs = np.where(mask)
            y_min, y_max = ys.min(), ys.max()
            x_min, x_max = xs.min(), xs.max()
            arr = arr[y_min:y_max+1, x_min:x_max+1, :]

    # Convertimos a RGB sobre fondo negro
    rgb = arr[:, :, :3].astype(np.float32)
    alpha = arr[:, :, 3:4].astype(np.float32) / 255.0
    rgb = rgb * alpha  # fondo negro implícito

    rgb_img = Image.fromarray(np.uint8(np.clip(rgb, 0, 255)))
    rgb_img = rgb_img.resize(size)

    thumb = np.array(rgb_img).astype(np.float32) / 255.0
    return thumb

In [ ]:
# ============================================================
# 7.5 Funcion para calcular similitud intra-clase simple
# ============================================================
# Usamos una similitud sencilla basada en MSE:
#
# similarity = 1 / (1 + mse)
#
# Cuanto mas parecidas dos imagenes, mayor similitud.
#
# OJO:
# Esta metrica es muy cruda y sensible a pose/escala.
# No es una metrica de rendimiento del modelo.
# Solo la usamos para EDA exploratorio.
# ============================================================

def simple_image_similarity(a, b):
    mse = np.mean((a - b) ** 2)
    sim = 1.0 / (1.0 + mse)
    return sim


def compute_intra_class_similarity(identity, max_images=20, thumb_size=(64, 64), use_alpha_bbox=True):
    """
    Calcula similitud simple entre todos los pares dentro de una identidad.

    Parametros:
    - identity: nombre del jaguar
    - max_images: maximo numero de imagenes a usar por identidad
    - thumb_size: tamano del thumbnail
    - use_alpha_bbox: si True, recorta con alpha antes del resize

    Retorna:
    - dict con resumen de similitudes intra-clase
    """
    subset = train_df[train_df["ground_truth"] == identity].copy()

    # Limitamos el numero para mantener el costo bajo
    if len(subset) > max_images:
        subset = subset.sample(n=max_images, random_state=42)

    thumbs = []
    filenames = []

    for _, row in subset.iterrows():
        thumb = make_thumbnail_array(
            row["full_path"],
            size=thumb_size,
            use_alpha_bbox=use_alpha_bbox
        )
        thumbs.append(thumb)
        filenames.append(row["filename"])

    sims = []
    pair_names = []

    for i in range(len(thumbs)):
        for j in range(i + 1, len(thumbs)):
            sim = simple_image_similarity(thumbs[i], thumbs[j])
            sims.append(sim)
            pair_names.append((filenames[i], filenames[j]))

    sims = np.array(sims)

    return {
        "ground_truth": identity,
        "n_images_used": len(thumbs),
        "n_pairs": len(sims),
        "sim_mean": sims.mean() if len(sims) > 0 else np.nan,
        "sim_std": sims.std() if len(sims) > 0 else np.nan,
        "sim_min": sims.min() if len(sims) > 0 else np.nan,
        "sim_25": np.percentile(sims, 25) if len(sims) > 0 else np.nan,
        "sim_50": np.percentile(sims, 50) if len(sims) > 0 else np.nan,
        "sim_75": np.percentile(sims, 75) if len(sims) > 0 else np.nan,
        "sim_max": sims.max() if len(sims) > 0 else np.nan,
    }

In [ ]:
# ============================================================
# 7.6 Resumen de similitud intra-clase para identidades seleccionadas
# ============================================================

intra_results = []

for identity in selected_identities:
    result = compute_intra_class_similarity(
        identity=identity,
        max_images=20,
        thumb_size=(64, 64),
        use_alpha_bbox=True
    )
    intra_results.append(result)

intra_df = pd.DataFrame(intra_results).sort_values("sim_mean", ascending=False)

print("Resumen de similitud intra-clase (metrica visual simple):")
display(intra_df)

In [ ]:
# ============================================================
# 7.7 Graficos para interpretar variabilidad intra-clase
# ============================================================
# Interpretacion aproximada:
# - sim_mean alta  -> muchas imagenes visualmente parecidas
# - sim_mean baja  -> identidad mas diversa
# ============================================================

plt.figure(figsize=(12, 5))
plt.bar(intra_df["ground_truth"], intra_df["sim_mean"])
plt.title("Similitud intra-clase media (aproximacion visual simple)", fontsize=14)
plt.xlabel("Identidad")
plt.ylabel("sim_mean")
plt.xticks(rotation=45)
plt.show()

plt.figure(figsize=(12, 5))
plt.bar(intra_df["ground_truth"], intra_df["sim_std"])
plt.title("Dispersion de similitud intra-clase", fontsize=14)
plt.xlabel("Identidad")
plt.ylabel("sim_std")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# ============================================================
# 7.8 Identidades mas repetitivas vs mas diversas
# ============================================================

print("Identidades mas repetitivas visualmente (segun esta aproximacion):")
display(intra_df.sort_values("sim_mean", ascending=False).head(5))

print("Identidades mas diversas visualmente (segun esta aproximacion):")
display(intra_df.sort_values("sim_mean", ascending=True).head(5))

In [ ]:
# ============================================================
# 7.9 Analisis global para todas las identidades (ligero)
# ============================================================
# Esto ya da una foto mas completa del dataset.
# Para mantenerlo liviano:
# - max 12 imagenes por identidad
# - thumbnails pequenos
# ============================================================

all_identity_results = []

for identity in class_dist_df["ground_truth"]:
    result = compute_intra_class_similarity(
        identity=identity,
        max_images=12,
        thumb_size=(48, 48),
        use_alpha_bbox=True
    )
    all_identity_results.append(result)

all_intra_df = pd.DataFrame(all_identity_results).sort_values("sim_mean", ascending=False)

print("Resumen global de variabilidad intra-clase:")
display(all_intra_df.head(10))
display(all_intra_df.tail(10))

In [ ]:
# ============================================================
# 7.10 Relacion entre numero de imagenes y similitud intra-clase
# ============================================================
# Esto puede sugerir:
# - clases con muchas imagenes pero muy repetitivas
# - clases con pocas imagenes y mucha variabilidad
# ============================================================

merged_intra = class_dist_df.merge(
    all_intra_df[["ground_truth", "sim_mean", "sim_std", "n_images_used"]],
    on="ground_truth",
    how="left"
)

plt.figure(figsize=(7, 6))
plt.scatter(merged_intra["n_images"], merged_intra["sim_mean"], alpha=0.7)
plt.title("Numero de imagenes vs similitud intra-clase media", fontsize=14)
plt.xlabel("n_images")
plt.ylabel("sim_mean")
plt.show()

display(merged_intra.sort_values("sim_mean", ascending=False).head(10))

# SIMILITUD INTER CLASE

In [ ]:
# ============================================================
# JAGUAR RE-IDENTIFICATION CHALLENGE - EDA (PARTE 8)
# SIMILITUD INTER-CLASE
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.notebook import tqdm
import math

In [ ]:
# ============================================================
# 8.1 Funciones auxiliares
# ============================================================

def make_thumbnail_array(img_path, size=(64, 64), use_alpha_bbox=True):
    """
    Crea un thumbnail RGB normalizado en [0,1].
    
    Si use_alpha_bbox=True, recorta la imagen al bounding box definido
    por alpha > 0 antes de redimensionar.
    """
    img = Image.open(img_path).convert("RGBA")
    arr = np.array(img)

    if use_alpha_bbox:
        alpha = arr[:, :, 3]
        mask = alpha > 0
        if mask.any():
            ys, xs = np.where(mask)
            y_min, y_max = ys.min(), ys.max()
            x_min, x_max = xs.min(), xs.max()
            arr = arr[y_min:y_max+1, x_min:x_max+1, :]

    rgb = arr[:, :, :3].astype(np.float32)
    alpha = arr[:, :, 3:4].astype(np.float32) / 255.0

    # Fondo negro implicito usando alpha
    rgb = rgb * alpha

    rgb_img = Image.fromarray(np.uint8(np.clip(rgb, 0, 255)))
    rgb_img = rgb_img.resize(size)

    thumb = np.array(rgb_img).astype(np.float32) / 255.0
    return thumb


def simple_image_similarity(a, b):
    """
    Similitud sencilla basada en MSE.
    Cuanto mas parecidas las imagenes, mayor score.
    """
    mse = np.mean((a - b) ** 2)
    sim = 1.0 / (1.0 + mse)
    return sim

In [ ]:
# ============================================================
# 8.2 Crear thumbnails por identidad
# ============================================================
# Para no hacer esto pesado:
# - usamos max_images_per_identity
# - thumbnails pequenos
# ============================================================

max_images_per_identity = 8
thumb_size = (64, 64)

identity_to_thumbs = {}
identity_to_rows = {}

for identity in tqdm(class_dist_df["ground_truth"].tolist(), desc="Creando thumbnails por identidad"):
    subset = train_df[train_df["ground_truth"] == identity].copy()

    if len(subset) > max_images_per_identity:
        subset = subset.sample(n=max_images_per_identity, random_state=42)

    thumbs = []
    rows = []

    for _, row in subset.iterrows():
        thumb = make_thumbnail_array(
            row["full_path"],
            size=thumb_size,
            use_alpha_bbox=True
        )
        thumbs.append(thumb)
        rows.append(row)

    identity_to_thumbs[identity] = thumbs
    identity_to_rows[identity] = rows

print("Numero de identidades procesadas:", len(identity_to_thumbs))

In [ ]:
# ============================================================
# 8.3 Similitud entre identidades
# ============================================================
# Para cada par de identidades:
# - comparamos todos los thumbnails entre ambas
# - resumimos con media, maximo y mediana
#
# Esto nos da una nocion de que tan parecidas parecen dos clases.
# ============================================================

identities = class_dist_df["ground_truth"].tolist()

pair_results = []

for i in tqdm(range(len(identities)), desc="Calculando similitud inter-clase"):
    id_a = identities[i]
    thumbs_a = identity_to_thumbs[id_a]

    for j in range(i + 1, len(identities)):
        id_b = identities[j]
        thumbs_b = identity_to_thumbs[id_b]

        sims = []

        for ta in thumbs_a:
            for tb in thumbs_b:
                sim = simple_image_similarity(ta, tb)
                sims.append(sim)

        sims = np.array(sims)

        pair_results.append({
            "identity_a": id_a,
            "identity_b": id_b,
            "n_pairs": len(sims),
            "sim_mean": sims.mean(),
            "sim_median": np.median(sims),
            "sim_max": sims.max(),
            "sim_min": sims.min(),
            "sim_std": sims.std()
        })

inter_df = pd.DataFrame(pair_results)

print("Shape de inter_df:", inter_df.shape)
display(inter_df.head())

In [ ]:
# ============================================================
# 8.4 Top pares de identidades mas parecidas
# ============================================================

top_similar_pairs = inter_df.sort_values("sim_mean", ascending=False).head(15)

print("Top 15 pares de identidades mas parecidas (segun similitud visual simple):")
display(top_similar_pairs)

In [ ]:
# ============================================================
# 8.5 Top pares menos parecidos
# ============================================================

least_similar_pairs = inter_df.sort_values("sim_mean", ascending=True).head(15)

print("Top 15 pares de identidades menos parecidas:")
display(least_similar_pairs)

In [ ]:
# ============================================================
# 8.6 Construir matriz de similitud inter-clase
# ============================================================

sim_matrix = pd.DataFrame(
    np.eye(len(identities)),
    index=identities,
    columns=identities
)

for _, row in inter_df.iterrows():
    a = row["identity_a"]
    b = row["identity_b"]
    sim = row["sim_mean"]

    sim_matrix.loc[a, b] = sim
    sim_matrix.loc[b, a] = sim

display(sim_matrix.head())

In [ ]:
# ============================================================
# 8.7 Heatmap de similitud inter-clase
# ============================================================

plt.figure(figsize=(14, 12))
sns.heatmap(sim_matrix, cmap="viridis")
plt.title("Heatmap de similitud inter-clase (sim_mean)", fontsize=16)
plt.xlabel("Identidad")
plt.ylabel("Identidad")
plt.show()

In [ ]:
# ============================================================
# 8.8 Visualizar pares de identidades muy parecidas
# ============================================================

def show_two_identities_side_by_side(identity_a, identity_b, n=4, seed=42):
    """
    Muestra n imagenes de identity_a y n imagenes de identity_b lado a lado.
    Muy util para inspeccion visual de clases parecidas.
    """
    subset_a = train_df[train_df["ground_truth"] == identity_a].copy()
    subset_b = train_df[train_df["ground_truth"] == identity_b].copy()

    n_a = min(n, len(subset_a))
    n_b = min(n, len(subset_b))

    subset_a = subset_a.sample(n=n_a, random_state=seed)
    subset_b = subset_b.sample(n=n_b, random_state=seed)

    fig, axes = plt.subplots(2, max(n_a, n_b), figsize=(4 * max(n_a, n_b), 8))

    if max(n_a, n_b) == 1:
        axes = np.array(axes).reshape(2, 1)

    for ax in axes.flatten():
        ax.axis("off")

    for idx, (_, row) in enumerate(subset_a.iterrows()):
        img = Image.open(row["full_path"]).convert("RGBA")
        axes[0, idx].imshow(img)
        axes[0, idx].set_title(f"{identity_a}\n{row['filename']}", fontsize=9)
        axes[0, idx].axis("off")

    for idx, (_, row) in enumerate(subset_b.iterrows()):
        img = Image.open(row["full_path"]).convert("RGBA")
        axes[1, idx].imshow(img)
        axes[1, idx].set_title(f"{identity_b}\n{row['filename']}", fontsize=9)
        axes[1, idx].axis("off")

    plt.suptitle(f"Comparacion visual: {identity_a} vs {identity_b}", fontsize=16)
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# 8.9 Inspeccionar visualmente los pares mas parecidos
# ============================================================
# Revisa manualmente estos pares porque aqui pueden aparecer:
# - clases genuinamente parecidas
# - sesgos de pose/composicion
# - limitaciones de la metrica simple
# ============================================================

pairs_to_review = top_similar_pairs[["identity_a", "identity_b"]].head(5).values.tolist()

for identity_a, identity_b in pairs_to_review:
    show_two_identities_side_by_side(identity_a, identity_b, n=4, seed=42)

In [ ]:
# ============================================================
# 8.10 Resumen por identidad: cual es su vecino mas parecido
# ============================================================

identity_neighbor_rows = []

for identity in identities:
    temp = inter_df[
        (inter_df["identity_a"] == identity) | (inter_df["identity_b"] == identity)
    ].copy()

    def get_other_identity(row, current_identity):
        return row["identity_b"] if row["identity_a"] == current_identity else row["identity_a"]

    temp["other_identity"] = temp.apply(lambda row: get_other_identity(row, identity), axis=1)
    temp = temp.sort_values("sim_mean", ascending=False)

    best_match = temp.iloc[0]

    identity_neighbor_rows.append({
        "ground_truth": identity,
        "most_similar_identity": best_match["other_identity"],
        "sim_mean": best_match["sim_mean"],
        "sim_median": best_match["sim_median"],
        "sim_max": best_match["sim_max"]
    })

identity_neighbors_df = pd.DataFrame(identity_neighbor_rows).sort_values("sim_mean", ascending=False)

print("Vecino visual mas parecido para cada identidad:")
display(identity_neighbors_df)

In [ ]:
# ============================================================
# 8.11 Grafico: que identidades tienen vecino mas parecido mas alto
# ============================================================

plt.figure(figsize=(14, 5))
plt.bar(identity_neighbors_df["ground_truth"], identity_neighbors_df["sim_mean"])
plt.title("Similitud con el vecino inter-clase mas parecido", fontsize=14)
plt.xlabel("Identidad")
plt.ylabel("sim_mean del vecino mas parecido")
plt.xticks(rotation=90)
plt.show()